# Getting Started with CTSM and NEON Tower Sites

This notebook walks you through the containerized CTSM environment and
confirms that everything is set up correctly on your machine. You can
run it immediately after launching the container; no credentials or data
downloads are needed.

By the end you will have:
- Verified the Python analysis stack (xarray, cartopy, etc.)
- Written and read a NetCDF file
- Rendered a projected map
- Confirmed that the CTSM source tree and CIME tools are accessible
- Seen which NEON tower sites are available
- Created a sample CLM case at Konza Prairie

If you are already comfortable with CTSM and CIME, this is a quick
sanity check. If you are new to the NEON tower workflow, the
explanations in each section provide context for the tools and
terminology you will encounter in the project's science notebooks.

## 1. Python Analysis Environment

The container ships a pre-configured Python environment with the
standard tools for working with CLM output. This cell imports the key
packages and prints their versions. If anything fails here, the
container image may be corrupted or incomplete.

In [ ]:
import sys
print(f"Python {sys.version}")
print(f"Platform: {sys.platform}")
print()

packages = [
    "numpy", "scipy", "pandas", "xarray", "netCDF4",
    "matplotlib", "cartopy", "bokeh", "holoviews", "panel",
    "jupyterlab", "dask", "boto3", "esmpy",
]

for name in packages:
    mod = __import__(name)
    ver = getattr(mod, "__version__", "ok")
    print(f"  {name:15s} {ver}")

print("\nAll imports OK.")

## 2. NetCDF I/O

CLM writes its output (history files) in NetCDF format. This cell
creates a small synthetic temperature field, writes it to a NetCDF file,
and reads it back to verify that the underlying I/O libraries (HDF5,
NetCDF-C) are linked correctly.

The project notebooks use xarray throughout for loading and manipulating
CLM history files. If this round-trip works, xarray and its dependencies
are healthy.

In [ ]:
import numpy as np
import xarray as xr
import tempfile, pathlib

# Create a small synthetic dataset
lats = np.linspace(-90, 90, 19)
lons = np.linspace(0, 360, 36, endpoint=False)
np.random.seed(42)
data = 260 + 30 * np.random.rand(19, 36).astype(np.float32)

ds = xr.Dataset(
    {"temperature": (["lat", "lon"], data)},
    coords={"lat": lats, "lon": lons},
    attrs={"title": "Smoke test dataset"},
)

# Write to NetCDF and read back
with tempfile.TemporaryDirectory() as tmp:
    path = pathlib.Path(tmp) / "test.nc"
    ds.to_netcdf(path)
    file_size = path.stat().st_size
    ds_read = xr.open_dataset(path)
    assert "temperature" in ds_read
    np.testing.assert_array_almost_equal(
        ds_read["temperature"].values, data, decimal=5
    )
    ds_read.close()

print(f"NetCDF round-trip OK ({file_size:,} bytes written).")
ds

## 3. Map Visualization

Cartopy handles map projections and geospatial plotting. The project
notebooks use it to render CLM output fields (soil temperature, moisture,
carbon fluxes) on geographic maps and to mark NEON tower locations.

This cell renders a simple two-panel figure: coastlines on the left, and
the synthetic temperature field from the previous cell projected onto a
Robinson globe on the right. The first run may take a few seconds while
cartopy downloads its coastline shapefiles.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig, axes = plt.subplots(
    1, 2, figsize=(14, 4),
    subplot_kw={"projection": ccrs.Robinson()},
)

# Left: coastlines and borders
ax = axes[0]
ax.set_global()
ax.add_feature(cfeature.LAND, facecolor="#e8e8e8")
ax.add_feature(cfeature.OCEAN, facecolor="#d0e4f0")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle="--")
ax.set_title("Coastlines + Borders")

# Right: pcolormesh of the synthetic temperature data
ax = axes[1]
ax.set_global()
lon2d, lat2d = np.meshgrid(lons, lats)
im = ax.pcolormesh(
    lon2d, lat2d, data,
    transform=ccrs.PlateCarree(),
    cmap="RdYlBu_r", vmin=260, vmax=290,
)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.set_title("Synthetic Temperature Field")
plt.colorbar(im, ax=ax, orientation="horizontal", pad=0.05, label="K")

plt.tight_layout()
plt.show()
print("Cartopy rendering OK.")

## 4. CTSM Source Tree and CIME

The container includes the complete CTSM 5.4 source tree at
`/opt/ncar/ctsm`. CIME (Common Infrastructure for Modeling the Earth)
lives inside it and provides the case management workflow:

| Command | What it does |
|---------|-------------|
| `create_newcase` | Set up a new simulation directory for a given compset, resolution, and machine |
| `case.setup` | Generate namelists and build scripts for the case |
| `case.build` | Compile the Fortran model (~2 min on this container) |
| `case.submit` | Execute the simulation |

The project's `run_neon_v2` wrapper calls these CIME commands internally,
so you rarely invoke them directly. This cell just verifies the tools
are accessible.

In [ ]:
import os
import subprocess

ctsm_root = os.environ.get("CESMROOT", "/opt/ncar/ctsm")
print(f"CTSM root:    {ctsm_root}")
print(f"CIME machine: {os.environ.get('CIME_MACHINE', 'not set')}")
print()

# Verify key scripts exist
for script in ["create_newcase", "query_config"]:
    path = os.path.join(ctsm_root, "cime", "scripts", script)
    exists = os.path.exists(path)
    print(f"  {script:20s} {'OK' if exists else 'MISSING'}")

# Verify CTSM Python modules
from ctsm import add_cime_to_path
from ctsm.path_utils import path_to_ctsm_root
print(f"\n  ctsm.path_to_ctsm_root() = {path_to_ctsm_root()}")
print("\nCTSM/CIME availability OK.")

## 5. Available NEON Tower Sites

Each NEON tower site has a **usermods** directory in the CTSM source
tree containing its grid cell location, surface dataset references,
and CLM namelist overrides. When you create a case for a NEON site,
CIME applies these usermods to configure CLM for that specific tower's
location, soil type, and vegetation.

This cell scans the usermods directory and lists all available sites.
The four-letter codes are standard NEON site abbreviations (e.g., KONZ
= Konza Prairie, HARV = Harvard Forest, ORNL = Oak Ridge). You can look
up any site on the
[NEON field sites map](https://www.neonscience.org/field-sites/explore-field-sites).

In [ ]:
import glob

neon_dir = os.path.join(ctsm_root, "cime_config", "usermods_dirs", "clm", "NEON")
sites = sorted([
    os.path.basename(d)
    for d in glob.glob(os.path.join(neon_dir, "[!d]*"))
    if os.path.isdir(d)
])

print(f"NEON usermods directory: {neon_dir}")
print(f"Sites found: {len(sites)}")
print()

# Display in columns
cols = 8
for i in range(0, len(sites), cols):
    print("  ".join(f"{s:6s}" for s in sites[i:i+cols]))

assert len(sites) >= 40, f"Expected 40+ NEON sites, found {len(sites)}"
print(f"\nNEON site discovery OK ({len(sites)} sites).")

## 6. Creating a NEON Case

This cell creates a CLM case for **KONZ** (Konza Prairie Biological
Station, Kansas), a tallgrass prairie Long-Term Ecological Research
site and one of the most-studied NEON locations.

The case uses the `I1PtClm60Bgc` compset (single-point CLM 6.0 with
biogeochemistry) at the `CLM_USRDAT` resolution (user-defined, set by
the KONZ usermods). Only `case.setup` is run here, which generates the
namelists and build scripts without downloading input data or compiling
Fortran.

In the project's science notebooks, `run_neon_v2` handles all of this
automatically when you specify a site code.

In [ ]:
%%bash
set -e

SITE="KONZ"
OUTPUT_ROOT="/tmp/smoke_test_neon"
rm -rf "$OUTPUT_ROOT"
mkdir -p /home/user/inputdata /home/user/scratch

echo "Creating NEON case for site: $SITE"
echo "Output root: $OUTPUT_ROOT"
echo

# Create case (setup only, no build or run)
$CESMROOT/cime/scripts/create_newcase \
    --case "$OUTPUT_ROOT/$SITE" \
    --compset I1PtClm60Bgc \
    --res CLM_USRDAT \
    --machine container \
    --run-unsupported \
    --user-mods-dirs "$CESMROOT/cime_config/usermods_dirs/clm/NEON/$SITE" \
    2>&1 | tail -5

echo
echo "Running case.setup..."
cd "$OUTPUT_ROOT/$SITE" && ./case.setup 2>&1 | tail -3

echo
echo "Case directory contents:"
ls "$OUTPUT_ROOT/$SITE/" | head -10

echo
echo "NEON case creation OK."

# Clean up
rm -rf "$OUTPUT_ROOT"

## Next Steps

The environment is verified. From here:

- **Run a NEON tower simulation.** The `Design_Hub_v2` and
  `Modeling_Hub` notebooks run CLM at NEON sites with forcing
  perturbations (e.g., scaled precipitation, temperature offsets) and
  compare output against tower observations. These require S3
  credentials for forcing data.

- **Analyze existing model output.** The `Data_Hub` notebook loads
  pre-computed CLM history files from cloud storage, computes soil
  temperature and moisture profiles, and plots time series against
  NEON observations.

- **Explore CTSM further.** The
  [NCAR CTSM Tutorial](https://github.com/NCAR/CTSM-Tutorial)
  includes hands-on notebooks covering NEON single-point runs, BGC
  diagnostics, and FATES vegetation dynamics. The
  [CTSM User's Guide](https://escomp.github.io/CTSM/) documents
  model physics, namelist options, and output variables.

- **Compare with tower observations.** Browse the
  [NEON Data Portal](https://data.neonscience.org/) to download
  soil temperature, moisture, and eddy covariance flux measurements
  for any site, then compare against your CLM output.